# Regression Analysis Practice Notebook

This notebook is a hands-on companion to the Markdown file on **Regression Analysis**.  
It demonstrates core regression models used for prediction and effect estimation.

Topics covered:

1. Linear regression  
2. Multiple linear regression  
3. Polynomial regression  
4. Logistic regression  
5. Ridge regression  
6. Lasso regression  
7. Elastic Net  
8. Poisson regression  
9. Quantile regression  
10. Nonlinear regression  
11. Model evaluation metrics  

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import (
    LinearRegression,
    LogisticRegression,
    Ridge,
    Lasso,
    ElasticNet,
    PoissonRegressor,
    QuantileRegressor,
)
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from scipy.optimize import curve_fit

np.random.seed(42)

## Create Example Datasets

We generate several small datasets for continuous regression, binary classification, count prediction, quantile regression, and nonlinear curve fitting.

In [ ]:
# 1D regression dataset
x = np.linspace(0, 10, 120)
y = 3 + 2.5 * x + np.random.normal(0, 2.5, len(x))

# Multiple regression dataset
n = 180
X_multi = pd.DataFrame({
    'x1': np.random.normal(10, 3, n),
    'x2': np.random.normal(20, 5, n),
    'x3': np.random.normal(5, 2, n)
})
y_multi = 4 + 1.8 * X_multi['x1'] - 0.9 * X_multi['x2'] + 2.2 * X_multi['x3'] + np.random.normal(0, 2, n)

# Binary classification dataset for logistic regression
X_log = pd.DataFrame({
    'f1': np.random.normal(0, 1, n),
    'f2': np.random.normal(0, 1, n)
})
logit_signal = 1.5 * X_log['f1'] - 1.0 * X_log['f2']
prob = 1 / (1 + np.exp(-logit_signal))
y_log = (np.random.rand(n) < prob).astype(int)

# Count dataset for Poisson regression
x_count = np.random.uniform(0, 4, n)
lam = np.exp(0.5 + 0.35 * x_count)
y_count = np.random.poisson(lam)

# Quantile regression dataset with heteroscedastic noise
x_q = np.linspace(0, 10, 160)
noise_q = np.random.normal(0, 0.8 + 0.4 * x_q, len(x_q))
y_q = 5 + 1.2 * x_q + noise_q

# Nonlinear dataset
x_non = np.linspace(0, 5, 100)
y_non = 2.5 * np.exp(0.6 * x_non) + np.random.normal(0, 2.0, len(x_non))

X_multi.head()

## 1. Linear Regression

Linear regression models a continuous response as:

$$
y_i = \beta_0 + \beta_1 x_i + \varepsilon_i
$$

In [ ]:
X_lin = x.reshape(-1, 1)
lin_model = LinearRegression()
lin_model.fit(X_lin, y)
y_pred_lin = lin_model.predict(X_lin)

pd.DataFrame({
    'Coefficient': [lin_model.coef_[0]],
    'Intercept': [lin_model.intercept_],
    'R2': [r2_score(y, y_pred_lin)]
})

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(x, y)
plt.plot(x, y_pred_lin)
plt.title('Linear Regression')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

## 2. Multiple Linear Regression

Multiple regression extends linear regression to multiple predictors:

$$
y_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \cdots + \beta_p x_{ip} + \varepsilon_i
$$

In [ ]:
mlr_model = LinearRegression()
mlr_model.fit(X_multi, y_multi)
y_pred_multi = mlr_model.predict(X_multi)

coef_table = pd.DataFrame({
    'Feature': X_multi.columns,
    'Coefficient': mlr_model.coef_
})
coef_table, pd.DataFrame({'Intercept': [mlr_model.intercept_], 'R2': [r2_score(y_multi, y_pred_multi)]})

## 3. Polynomial Regression

Polynomial regression models curved relationships:

$$
y = \beta_0 + \beta_1 x + \beta_2 x^2 + \cdots + \beta_d x^d + \varepsilon
$$

In [ ]:
y_curve = 4 + 1.5 * x - 0.25 * x**2 + np.random.normal(0, 2.0, len(x))
poly_model = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())
poly_model.fit(X_lin, y_curve)
y_pred_poly = poly_model.predict(X_lin)

pd.DataFrame({'R2': [r2_score(y_curve, y_pred_poly)]})

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(x, y_curve)
plt.plot(x, y_pred_poly)
plt.title('Polynomial Regression (Degree 2)')
plt.xlabel('x')
plt.ylabel('y_curve')
plt.show()

## 4. Logistic Regression

Logistic regression models binary outcomes through the logit function:

$$
P(Y=1|X)=\frac{1}{1+\exp[-(\beta_0+\beta^T X)]}
$$

In [ ]:
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X_log, y_log, test_size=0.3, random_state=42)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_log, y_train_log)
y_pred_log = log_model.predict(X_test_log)
y_prob_log = log_model.predict_proba(X_test_log)[:, 1]

pd.DataFrame({
    'Metric': ['Accuracy'],
    'Value': [accuracy_score(y_test_log, y_pred_log)]
}), pd.DataFrame(confusion_matrix(y_test_log, y_pred_log), columns=['Pred 0', 'Pred 1'], index=['True 0', 'True 1'])

## 5. Ridge Regression

Ridge regression adds an L2 penalty:

$$
\min_\beta \sum (y_i-\hat{y}_i)^2 + \lambda \sum \beta_j^2
$$

In [ ]:
ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
ridge_model.fit(X_multi, y_multi)
y_pred_ridge = ridge_model.predict(X_multi)
pd.DataFrame({'Model': ['Ridge'], 'R2': [r2_score(y_multi, y_pred_ridge)], 'RMSE': [mean_squared_error(y_multi, y_pred_ridge) ** 0.5]})

## 6. Lasso Regression

Lasso regression adds an L1 penalty:

$$
\min_\beta \sum (y_i-\hat{y}_i)^2 + \lambda \sum |\beta_j|
$$

It can set some coefficients exactly to zero.

In [ ]:
lasso_model = make_pipeline(StandardScaler(), Lasso(alpha=0.1, max_iter=10000))
lasso_model.fit(X_multi, y_multi)
y_pred_lasso = lasso_model.predict(X_multi)
pd.DataFrame({'Model': ['Lasso'], 'R2': [r2_score(y_multi, y_pred_lasso)], 'RMSE': [mean_squared_error(y_multi, y_pred_lasso) ** 0.5]})

## 7. Elastic Net

Elastic Net combines L1 and L2 penalties:

$$
\min_\beta \sum (y_i-\hat{y}_i)^2 + \lambda_1 \sum |\beta_j| + \lambda_2 \sum \beta_j^2
$$

In [ ]:
enet_model = make_pipeline(StandardScaler(), ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000))
enet_model.fit(X_multi, y_multi)
y_pred_enet = enet_model.predict(X_multi)
pd.DataFrame({'Model': ['Elastic Net'], 'R2': [r2_score(y_multi, y_pred_enet)], 'RMSE': [mean_squared_error(y_multi, y_pred_enet) ** 0.5]})

## Compare Coefficients Across Linear, Ridge, Lasso, and Elastic Net

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_multi)

ols = LinearRegression().fit(X_scaled, y_multi)
ridge = Ridge(alpha=1.0).fit(X_scaled, y_multi)
lasso = Lasso(alpha=0.1, max_iter=10000).fit(X_scaled, y_multi)
enet = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000).fit(X_scaled, y_multi)

coef_compare = pd.DataFrame({
    'Feature': X_multi.columns,
    'OLS': ols.coef_,
    'Ridge': ridge.coef_,
    'Lasso': lasso.coef_,
    'ElasticNet': enet.coef_
})
coef_compare

## 8. Poisson Regression

Poisson regression is used for count data:

$$
Y_i \sim Poisson(\lambda_i), \quad \log(\lambda_i)=\beta_0+\beta^T X_i
$$

In [ ]:
X_count = x_count.reshape(-1, 1)
pois_model = PoissonRegressor(alpha=0.0, max_iter=1000)
pois_model.fit(X_count, y_count)
y_pred_count = pois_model.predict(X_count)

pd.DataFrame({
    'Coefficient': [pois_model.coef_[0]],
    'Intercept': [pois_model.intercept_],
    'MAE': [mean_absolute_error(y_count, y_pred_count)]
})

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(x_count, y_count)
order = np.argsort(x_count)
plt.plot(x_count[order], y_pred_count[order])
plt.title('Poisson Regression')
plt.xlabel('x_count')
plt.ylabel('y_count')
plt.show()

## 9. Quantile Regression

Quantile regression models a conditional quantile:

$$
Q_\tau(Y|X)=\beta_0+\beta^T X
$$

It is useful for robust prediction and uncertainty bands.

In [ ]:
X_q = x_q.reshape(-1, 1)
q50_model = QuantileRegressor(quantile=0.5, alpha=0.0, solver='highs')
q90_model = QuantileRegressor(quantile=0.9, alpha=0.0, solver='highs')
q50_model.fit(X_q, y_q)
q90_model.fit(X_q, y_q)

y_q50 = q50_model.predict(X_q)
y_q90 = q90_model.predict(X_q)

pd.DataFrame({
    'Model': ['Quantile 0.5', 'Quantile 0.9'],
    'Slope': [q50_model.coef_[0], q90_model.coef_[0]],
    'Intercept': [q50_model.intercept_, q90_model.intercept_]
})

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(x_q, y_q)
plt.plot(x_q, y_q50, label='Quantile 0.5')
plt.plot(x_q, y_q90, label='Quantile 0.9')
plt.title('Quantile Regression')
plt.xlabel('x_q')
plt.ylabel('y_q')
plt.legend()
plt.show()

## 10. Nonlinear Regression

Nonlinear regression models relationships of the form:

$$
y=f(x,\theta)+\varepsilon
$$

Here we fit an exponential curve.

In [ ]:
def exp_model(x, a, b):
    return a * np.exp(b * x)

params, _ = curve_fit(exp_model, x_non, y_non, p0=[2.0, 0.5], maxfev=10000)
a_hat, b_hat = params
y_pred_non = exp_model(x_non, a_hat, b_hat)

pd.DataFrame({'Parameter': ['a', 'b', 'R2'], 'Value': [a_hat, b_hat, r2_score(y_non, y_pred_non)]})

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(x_non, y_non)
plt.plot(x_non, y_pred_non)
plt.title('Nonlinear Regression: Exponential Fit')
plt.xlabel('x_non')
plt.ylabel('y_non')
plt.show()

## 11. Model Evaluation Metrics

Common regression evaluation metrics are:

$$
MAE=\frac{1}{n}\sum |y_i-\hat{y}_i|
$$

$$
RMSE=\sqrt{\frac{1}{n}\sum (y_i-\hat{y}_i)^2}
$$

$$
R^2 = 1-\frac{\sum (y_i-\hat{y}_i)^2}{\sum (y_i-\bar{y})^2}
$$

In [ ]:
metric_summary = pd.DataFrame({
    'Model': ['Linear', 'Multiple Linear', 'Polynomial', 'Ridge', 'Lasso', 'ElasticNet', 'Poisson', 'Nonlinear'],
    'MAE': [
        mean_absolute_error(y, y_pred_lin),
        mean_absolute_error(y_multi, y_pred_multi),
        mean_absolute_error(y_curve, y_pred_poly),
        mean_absolute_error(y_multi, y_pred_ridge),
        mean_absolute_error(y_multi, y_pred_lasso),
        mean_absolute_error(y_multi, y_pred_enet),
        mean_absolute_error(y_count, y_pred_count),
        mean_absolute_error(y_non, y_pred_non),
    ],
    'RMSE': [
        mean_squared_error(y, y_pred_lin) ** 0.5,
        mean_squared_error(y_multi, y_pred_multi) ** 0.5,
        mean_squared_error(y_curve, y_pred_poly) ** 0.5,
        mean_squared_error(y_multi, y_pred_ridge) ** 0.5,
        mean_squared_error(y_multi, y_pred_lasso) ** 0.5,
        mean_squared_error(y_multi, y_pred_enet) ** 0.5,
        mean_squared_error(y_count, y_pred_count) ** 0.5,
        mean_squared_error(y_non, y_pred_non) ** 0.5,
    ],
    'R2': [
        r2_score(y, y_pred_lin),
        r2_score(y_multi, y_pred_multi),
        r2_score(y_curve, y_pred_poly),
        r2_score(y_multi, y_pred_ridge),
        r2_score(y_multi, y_pred_lasso),
        r2_score(y_multi, y_pred_enet),
        np.nan,
        r2_score(y_non, y_pred_non),
    ]
})
metric_summary

## 12. Mini Exercises

Try these on your own:

1. Change the polynomial degree from 2 to 3 or 4 and compare the fit.  
2. Increase the Ridge and Lasso penalty values and inspect how coefficients shrink.  
3. Create a dataset with stronger class separation for logistic regression.  
4. Compare quantile 0.1, 0.5, and 0.9 in quantile regression.  
5. Replace the synthetic data with your own engineering dataset and compare linear, ridge, lasso, and elastic net.  
6. Fit another nonlinear form such as a sigmoid or power law.

These exercises are especially useful for AI, machine learning, structural engineering, surrogate modeling, and scientific data analysis.